In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler # NEU
import joblib
from torch.utils.data import TensorDataset, DataLoader

# 1. Laden des Datensatzes
df = pd.read_csv('Datasets/Iris.csv')

# 2. Features (X) und Target (y) trennen
X = df.drop('species', axis=1).values
y = df['species'].values

# LabelEncoder sortiert die gefundenen Text-Kategorien standardmäßig alphabetisch
le = LabelEncoder()
y = le.fit_transform(y)
 
# 3. Aufteilen des Datensatzes (Train / Val / Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

# Feature Scaling ---
# Für neuronale Netze ist Skalierung essenziell für die Konvergenz. Gängige Methoden:
# 1. StandardScaler (Standardisierung): Mittelwert = 0, Standardabweichung = 1 (Standard für NNs)
# 2. MinMaxScaler (Normalisierung): Skaliert Werte starr in einen Bereich (meist 0 bis 1)
# 3. RobustScaler: Nutzt Median und Quartile, sehr robust gegenüber Ausreißern (Outliers)
scaler = StandardScaler()

# WICHTIG (Vermeidung von Data Leakage): 'fit_transform' NUR auf Trainingsdaten anwenden!
# Der Scaler lernt hier den Mittelwert und die Varianz der Trainingsdaten.
X_train = scaler.fit_transform(X_train)

# Auf Validierungs- und Testdaten NUR 'transform' anwenden!
# Sie werden mit den Parametern skaliert, die aus den Trainingsdaten gelernt wurden.
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
# -----------------------------------------
 
# 4. Umwandeln der skalierten Daten in PyTorch-Tensoren
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()

# Erstellen eines TensorDatasets und DataLoaders - Kombinieren von Features und Labels zu einem Dataset
train_dataset = TensorDataset(X_train, y_train)
# Erstellen des DataLoaders für das Training
batch_size = 12
# shuffle=True sorgt dafür, dass die Daten in jeder Epoche gemischt werden
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Definieren des neuronalen Netzes als Sequential-Modell
net = nn.Sequential(
    nn.Linear(4, 5),   # Eingabeschicht (4 Merkmale) -> Versteckte Schicht (10 Neuronen)
    nn.ReLU(),           # Aktivierungsfunktion: ReLU
    nn.Linear(5, 3)     # Versteckte Schicht (10 Neuronen) -> Ausgabeschicht (3 Klassen)
)
 
# Definieren des Verlustkriteriums und des Optimierers
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.01)

# Listen zum Speichern der Historie (ideal für spätere Plots, z.B. mit matplotlib)
history = {'train_loss': [], 'val_loss': []}

# Trainieren des neuronalen Netzes
net.train()  # Trainingsmodus aktivieren

for epoch in range(100):
    net.train()
    kumulierter_train_loss = 0.0 # Variable zum Aufsummieren des Batch-Losses

    for batch_data in train_loader: # Schleife über die Batches aus dem DataLoader
        batch_X = batch_data[0]
        batch_y = batch_data[1]
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        # Loss aufsummieren
        kumulierter_train_loss += loss.item()
    
    durchschnittlicher_train_loss = kumulierter_train_loss / len(train_loader)
    history['train_loss'].append(durchschnittlicher_train_loss)

    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val)
    
    history['val_loss'].append(val_loss.item())
    print(f'Epoch {epoch:3d} | Loss: {durchschnittlicher_train_loss:.4f} | Val Loss: {val_loss:.4f}')
 
# Auswerten des neuronalen Netzes auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)
 
# Abspeichern des trainierten Netzes
torch.save(net.state_dict(), 'Models/iris_net_NextStep3.pth')
joblib.dump(le, 'Models/label_encoder_NextStep3.pkl')
joblib.dump(scaler, 'Models/scaler_NextStep3.pkl')

Epoch   0 | Loss: 1.1013 | Val Loss: 1.0667
Epoch   1 | Loss: 1.0003 | Val Loss: 0.9945
Epoch   2 | Loss: 0.9166 | Val Loss: 0.9112
Epoch   3 | Loss: 0.8187 | Val Loss: 0.8201
Epoch   4 | Loss: 0.6970 | Val Loss: 0.7223
Epoch   5 | Loss: 0.5859 | Val Loss: 0.6272
Epoch   6 | Loss: 0.4862 | Val Loss: 0.5676
Epoch   7 | Loss: 0.4167 | Val Loss: 0.5212
Epoch   8 | Loss: 0.3660 | Val Loss: 0.4973
Epoch   9 | Loss: 0.3301 | Val Loss: 0.4553
Epoch  10 | Loss: 0.2994 | Val Loss: 0.4248
Epoch  11 | Loss: 0.2753 | Val Loss: 0.4073
Epoch  12 | Loss: 0.2586 | Val Loss: 0.4159
Epoch  13 | Loss: 0.2374 | Val Loss: 0.3709
Epoch  14 | Loss: 0.2184 | Val Loss: 0.3451
Epoch  15 | Loss: 0.2077 | Val Loss: 0.3366
Epoch  16 | Loss: 0.1918 | Val Loss: 0.3286
Epoch  17 | Loss: 0.1815 | Val Loss: 0.2995
Epoch  18 | Loss: 0.1673 | Val Loss: 0.2950
Epoch  19 | Loss: 0.1597 | Val Loss: 0.2983
Epoch  20 | Loss: 0.1493 | Val Loss: 0.2754
Epoch  21 | Loss: 0.1413 | Val Loss: 0.2637
Epoch  22 | Loss: 0.1345 | Val L

['Models/scaler_NextStep3.pkl']

In [2]:
# Wiederladen des State-Dicts
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(4, 5),    # Eingabeschicht (4 Merkmale) -> Versteckte Schicht (10 Neuronen)
    nn.ReLU(),          # Aktivierungsfunktion: ReLU
    nn.Linear(5, 3)     # Versteckte Schicht (10 Neuronen) -> Ausgabeschicht (3 Klassen)
)

net.load_state_dict(torch.load('Models/iris_net_NextStep3.pth', map_location=torch.device('cpu')))
le = joblib.load('Models/label_encoder_NextStep3.pkl')
scaler = joblib.load('Models/scaler_NextStep3.pkl')

for k, v in net.named_parameters():
    print(k,v)
net.eval()

0.weight Parameter containing:
tensor([[ 0.0879, -0.5344,  1.0401,  1.6223],
        [ 0.0409, -0.6903,  0.9880,  1.2702],
        [ 0.8082, -0.9362,  0.4884, -0.3996],
        [-0.1956,  0.1295,  0.1417,  0.1429],
        [-0.8003,  0.4302, -0.8679, -1.1801]], requires_grad=True)
0.bias Parameter containing:
tensor([-1.1721,  0.3931,  1.8161, -0.5209,  1.5004], requires_grad=True)
2.weight Parameter containing:
tensor([[-0.3358, -1.1931, -1.3044, -0.0848,  1.5179],
        [-2.5447,  0.1926,  1.2187,  0.1493, -0.4551],
        [ 2.4610,  0.8985, -0.7024, -0.0210, -1.1112]], requires_grad=True)
2.bias Parameter containing:
tensor([-0.0329,  0.5500, -0.6653], requires_grad=True)


Sequential(
  (0): Linear(in_features=4, out_features=5, bias=True)
  (1): ReLU()
  (2): Linear(in_features=5, out_features=3, bias=True)
)

In [3]:
# Vorhersage mit dem trainierten Netz

# Eingabe der vier Merkmale
print("Geben Sie die vier Merkmale ein:")
sepal_length = float(input("Sepal-Länge (cm): "))
sepal_width = float(input("Sepal-Breite (cm): "))
petal_length = float(input("Petal-Länge (cm): "))
petal_width = float(input("Petal-Breite (cm): "))

# Skalierung der Eingabewerte
import numpy as np
eingabe = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
eingabe_skaliert = scaler.transform(eingabe)

# Erstellen eines Tensors aus den Eingabewerten
inputs = torch.tensor(eingabe_skaliert, dtype=torch.float32)

# Vorhersage treffen
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

# Ausgabe der Klassifizierungsaussage
print("Klasse: ", le.inverse_transform([predicted.item()])[0])

Geben Sie die vier Merkmale ein:
Klasse:  setosa


In [5]:
# aus dem obigen Skript wird inputs übernommen, 
# die Vorhersagen werden als Wahrscheinlichkeiten ausgegeben

import torch.nn.functional as F

# Vorhersage des Netzes
with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)
    print("Vorhergesagte Klasse:", max_index.item())

# Klassenwahrscheinlichkeiten berechnen
probabilities = F.softmax(output, dim=1)
print("Klassenwahrscheinlichkeiten:", probabilities.numpy()[0])

Vorhergesagte Klasse: 0
Klassenwahrscheinlichkeiten: [9.9876982e-01 1.1856663e-03 4.4538603e-05]
